# Pipeline de Extracción de Datos de *Open-Meteo*

Este *notebook* implementa un flujo completo de extracción, transformación y carga (ETL) para obtener datos históricos climáticos de *Open-Meteo*. El objetivo es consolidar una base de datos sobre diversas mediciones climaticas en las principales capitales europeas.

Obtenemos los datos de la API de *Open-Meteo*. Para ello, debemos definir el rango de tiempo sobre el que queremos los datos (en horas), la longitud y latitud de la ciudad deseada y, finalmente, seleccionar las variables de las que pretendemos tener los datos. 

Requerimos de estos datos para entender el comportamiento de los contaminantes, pues es imprescindible considerar las condiciones meteorológicas. Por consiguiente, en esta sección seguimos las siguientes consideraciones:

* **Geolocalización precisa**: empleamos coordenadas exactas (latitud/longitud) de las capitales europeas para asegurar la coincidencia geográfica con las estaciones de la EEA.

* **Sincronización temporal**: configuramos la zona horaria en UTC para garantizar un alineamiento perfecto al fusionar estos datos con las mediciones de calidad del aire.

* **Estructura espejo**: guardamos los datos climáticos siguiendo la misma jerarquía de carpetas por país que los datos de contaminantes, facilitando la integración final del dataset.

Además, las variables que contendrá cada *dataset* por país son:

1. *time*: momento (día-mes-año-hora) en que se toman los datos.
2. *temperature_2m*: temperatura a 2 metros en grados (°C).
3. *relative_humidity_2m*: humedad relativa a 2 metros, en porcentaje.
4. *precipitation*: precipitación total (lluvia + nieve), en mm (milímetros).
5. *rain*: lluvia en mm.
6. *snowfall*: lluvia de la nieve en cm.
7. *surface_pressure*: presión atmosférica en superficie, en hPa (hectopascal).
8. *cloudcover*: cobertura nubosa total, en porcentaje.
9. *windspeed_10m*: velocidad del viento a 10 metros, en km/h.
10. *shortwave_radiation*: radiación solar de onda corta, en $W/m^2$.
11. *boundary_layer_height*: capa PBLH (*Planetary Boundary Layer Height*) o **Altura de la Capa Límite Planetaria** en metros. Se podría comparar con un techo que tenemos todos los seres humanos y que permite que los gases producidos por vehículos o fábricas se vaya, por lo que se concentran.

In [1]:
# ==============================================================================
# IMPORTACIÓN DE LIBRERÍAS
# ==============================================================================
import requests           # Para realizar peticiones a la API climática.
import os    
import pandas as pd        # Procesamiento de las series temporales meteorológicas.
from pathlib import Path   # Gestión de rutas y directorios de forma robusta.
from tqdm import tqdm     # Visualización del progreso durante la descarga masiva.

In [2]:
# Endpoint de datos históricos (Archive API) de Open-Meteo.
BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

In [ ]:
# Definición del directorio de salida específico para datos climáticos.
BASE_DIR = Path(r"C:\Users\Juanfran cd\Desktop\Máster_Ciencia_de_Datos_UA\TFM\datasets\archivos_clima")
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Definimos la ruta raíz donde se encuentran las carpetas por país.
BASE_FOLDER = Path(r"C:\Users\Juanfran cd\Desktop\Máster_Ciencia_de_Datos_UA\TFM\datasets\archivos_clima")

In [4]:
# Rango temporal: debe coincidir con el periodo de los datos de la EEA (2013-2024).
START_DATE = "2013-01-01"
END_DATE   = "2024-12-31"

In [ ]:
# ==============================================================================
# CATÁLOGO DE COORDENADAS
# ==============================================================================
# Diccionario que mapea códigos ISO con las coordenadas (Latitud, Longitud) exactas obtenidas del notebook "países" de las capitales/países europeos analizados en el bloque de la EEA.

locations = {
    "AD": (42.546245, 1.601554),   "AL": (41.153332, 20.168331),
    "AT": (48.20849, 16.37208),    "BA": (43.915886, 17.679076),
    "BE": (50.85045, 4.34878),     "BG": (42.69751, 23.32415),
    "CH": (46.94809, 7.44744),     "CY": (35.17284, 33.35397),
    "CZ": (50.08804, 14.42076),    "DE": (52.52437, 13.41053),
    "DK": (55.67594, 12.56553),    "EE": (59.43696, 24.75353),
    "ES": (40.4165, -3.70256),     "FI": (60.16952, 24.93545),
    "FR": (48.85341, 2.3488),      "GR": (37.98376, 23.72784),
    "HR": (45.81444, 15.97798),    "HU": (47.49835, 19.04045),
    "IE": (53.33306, -6.24889),    "IS": (64.13548, -21.89541),
    "IT": (41.89193, 12.51133),    "LT": (54.68916, 25.2798),
    "LU": (49.815273, 6.129583),   "LV": (56.946, 24.10589),
    "ME": (42.708678, 19.37439),   "MK": (41.608635, 21.745275),
    "MT": (35.89968, 14.5148),     "NL": (52.37403, 4.88969),
    "NO": (59.91273, 10.74609),    "PL": (52.22977, 21.01178),
    "PT": (38.72509, -9.1498),     "RO": (44.43225, 26.10626),
    "RS": (44.016521, 21.005859),  "SE": (59.32938, 18.06871),
    "SI": (46.05108, 14.50513),    "SK": (48.14816, 17.10674),
    "XK": (42.602636, 20.902977)
}

In [ ]:
# ==============================================================================
# EJECUCIÓN DEL PIPELINE DE DESCARGA
# ==============================================================================

# Iteramos sobre cada ubicación para obtener su histórico meteorológico.
for country, (lat, lon) in tqdm(locations.items(), desc="Downloading climate data"):
    
    # Organización: Carpeta individual por país.
    country_dir = BASE_DIR / country
    country_dir.mkdir(exist_ok=True)

    # Definición de variables climáticas requeridas (Viento, Lluvia, Temperatura, etc.)
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": (
            "temperature_2m,"        
            "relative_humidity_2m," 
            "precipitation,"        
            "rain,"                  
            "weathercode,"          
            "surface_pressure,"      
            "cloudcover,"            
            "windspeed_10m,"         
            "shortwave_radiation",
            "boundary_layer_height"   
        ),
        "timezone": "UTC"            # Forzamos UTC para sincronizar con datos de la EEA.
    }

    # Solicitud HTTP GET con parámetros estructurados.
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()
    data = response.json()

    # Convertimos el bloque 'hourly' del JSON en una tabla.
    df = pd.DataFrame(data["hourly"])
    
    # Añadimos la columna de país como identificador primario para futuros joins.
    df.insert(0, "country", country)

    # Exportamos el DataFrame consolidado a CSV.
    output_file = country_dir / f"open-meteo-{country}.csv"
    df.to_csv(output_file, index=False)

print("\n✅ Todos los CSV climáticos han sido descargados y guardados correctamente.")


✅ Todos los CSV climáticos han sido descargados y guardados correctamente.


Los archivos descargados directamente de la API meteorológica contienen líneas de metadatos informativos al inicio que impiden la lectura directa mediante herramientas de análisis. Por tanto, realizamos una limpieza que:

* **Elimina los encabezados de la API**: se descartan las filas de metadatos técnicos (latitud, longitud, elevación) que no forman parte de la serie temporal.

* **Reindexación de columnas**: se redefine la cabecera del archivo para que la primera fila contenga exclusivamente los nombres de las variables climáticas.

* **Saneamiento de archivos**: sobrescribimos los archivos originales con un formato tabular puro, garantizando la compatibilidad.

In [ ]:
# ==============================================================================
# LIMPIEZA DE CABECERAS Y FORMATEO DE CSV METEOROLÓGICOS
# ==============================================================================

# Recorremos cada elemento dentro de la carpeta base.
for country_folder in BASE_FOLDER.iterdir():
    
    # Filtramos para procesar únicamente directorios (excluyendo archivos sueltos).
    if country_folder.is_dir(): 
        
        # Localizamos todos los archivos .csv dentro de la carpeta del país.
        for csv_file in country_folder.glob("*.csv"):
            print(f"Procesando {csv_file.name}...")
            
            try:
                # La API de Open-Meteo incluye metadatos en las primeras líneas (coordenadas, elevación, unidad de medida, etc.).
                # skiprows=3: ignoramos las primeras 3 líneas de metadatos informativos.
                # header=0: establecemos la cuarta línea como el encabezado real (nombres de columnas).
                df = pd.read_csv(csv_file, skiprows=3, header=0)

                # Sobrescribimos el archivo con la estructura limpia.
                # index=False evita añadir una columna numérica adicional.
                df.to_csv(csv_file, index=False)
                
            except Exception as e:
                # Captura de errores para evitar que el script se detenga si un archivo falla.
                print(f"Error procesando {csv_file.name}: {e}")

print("\n✅ Todos los CSV han sido limpiados y formateados correctamente.")


✅ Todos los CSV han sido limpiados y formateados correctamente.
